In [ ]:
import requests
import json
from notebookutils import mssparkutils
from pyspark.sql.functions import col
import time
from datetime import datetime, timezone

# Capture Start Time (for watermark)
execution_start_time = datetime.now(timezone.utc)
print(f"Execution Start Time: {execution_start_time}")

# --- Configuration ---
# Kusto Details
kusto_cluster_uri = "https://trd-6uegjpfbf030eemxtw.z1.kusto.fabric.microsoft.com"
kusto_database = "MonitoringEventhouse"

# Auth Details (Update with your SPN or Key Vault logic)
tenant_id = "your-tenant-id"
client_id = "your-client-id"
client_secret = "your-client-secret"

# Recommended: Use Key Vault
# client_secret = mssparkutils.credentials.getSecret("your-kv", "your-secret")

# API Base URL
fabric_api_base = "https://api.fabric.microsoft.com/v1"

StatementMeta(, f675129f-c741-4a05-934f-ca0367a2c63c, 15, Finished, Available, Finished, False)

Execution Start Time: 2026-03-04 14:20:28.767262+00:00


In [14]:
def get_spn_token(scope):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': scope
    }
    try:
        response = requests.post(url, data=payload)
        response.raise_for_status()
        return response.json().get("access_token")
    except Exception as e:
        print(f"Failed to retrieve SPN token for scope {scope}: {e}")
        return None

# 1. Get Fabric Token (for Delete API)
fabric_scope = "https://api.fabric.microsoft.com/.default" 
fabric_token = get_spn_token(fabric_scope)

# 2. Get Kusto Token (for Reading Data)
# Note: Kusto scope is typically {ClusterURI}/.default
kusto_scope = f"{kusto_cluster_uri}/.default"
kusto_token = get_spn_token(kusto_scope)

if fabric_token and kusto_token:
    print("Successfully retrieved tokens.")
else:
    print("Failed to retrieve essential tokens (Fabric/Kusto).")

StatementMeta(, f675129f-c741-4a05-934f-ca0367a2c63c, 16, Finished, Available, Finished, False)

Successfully retrieved tokens.


In [15]:
# --- Define Helper Functions ---

def delete_fabric_item(workspace_id, item_id, item_kind, max_retries=2):
    """
    Deletes a specific item from a Fabric Workspace.
    API: DELETE https://api.fabric.microsoft.com/v1/workspaces/{workspaceId}/{endpoint}/{itemId}
    Returns: (status_message, error_code, description)
    """
    if not fabric_token:
        print("Skipping Delete: No Fabric token available.")
        return "DeleteFailed", "NoToken", "No Fabric token available"

    # Map ItemKind to API endpoint segment
    # Default to 'items' if not found in list
    endpoint_map = {
        "ApacheAirflowJob": "ApacheAirflowJobs",
        "CopyJob": "copyJobs",
        "CosmosDBDatabase": "cosmosDbDatabases", # (no events coming)
        "DataFlow": "dataflows",
        "DataPipeline": "dataPipelines",
        "Environment": "environments",
        "Eventhouse": "eventhouses",
        "GraphQLApi": "GraphQLApis",
        "KQLDashboard": "kqlDashboards",
        "KQLQueryset": "kqlQuerysets",
        "Lakehouse": "lakehouses",
        "MirroredAzureDatabricksCatalog": "mirroredAzureDatabricksCatalogs",
        "MirroredDatabase": "mirroredDatabases",
        "MountedDataFactory": "mountedDataFactories",
        "Notebook": "notebooks",
        "Reflex": "reflexes",
        "SparkJobDefinition": "sparkJobDefinitions",
        "SQLDatabase": "sqlDatabases", # (no events coming)
        "UserDataFunction": "UserDataFunctions",
        "VariableLibrary": "VariableLibraries",
        "Warehouse": "warehouses"
    }

    # Use specific endpoint if mapped, otherwise default to generic 'items'
    endpoint = endpoint_map.get(item_kind, "items")
    url = f"{fabric_api_base}/workspaces/{workspace_id}/{endpoint}/{item_id}"
    
    headers = {
        "Authorization": f"Bearer {fabric_token}",
        "Content-Type": "application/json"
    }
    
    last_exception = None
    
    for attempt in range(max_retries):
        try:
            response = requests.delete(url, headers=headers)
            
            # 200 OK or 204 No Content are typical success codes for DELETE
            if response.status_code in [200, 204]:
                return "DeletedItem", str(response.status_code), "Success"
            elif response.status_code == 404:
                print(f"SKIPPED: Item {item_id} not found (already deleted?).")
                return "404: File Not Found", "404", "Item Not Found"
            elif response.status_code == 401:
                # Unauthorized - Need to add permissions
                print(f"UNAUTHORIZED (401): No permission for item {item_id} in ws {workspace_id}.")
                return "401: Unauthorized", "401", response.text
            elif response.status_code == 429:
                # Handle Rate Limiting
                print(f"RATE LIMIT (429) Headers: {response.headers}")
                retry_after = int(response.headers.get("Retry-After", 60))
                print(f"RATE LIMIT (429): Waiting {retry_after} seconds before retry {attempt + 1}/{max_retries}...")
                time.sleep(retry_after)
                continue
            else:
                print(f"FAILED: Could not delete item {item_id}. Status: {response.status_code}, Response: {response.text}")
                return "DeleteFailed", str(response.status_code), response.text
                
        except Exception as e:
            print(f"EXCEPTION: Error deleting item {item_id}: {e}")
            last_exception = e
    
    print(f"FAILED: Max retries ({max_retries}) exceeded for item {item_id}.")
    desc = str(last_exception) if last_exception else "Max retries exceeded"
    return "DeleteFailed", "MaxRetries", desc

StatementMeta(, f675129f-c741-4a05-934f-ca0367a2c63c, 17, Finished, Available, Finished, False)

In [16]:
# --- Read Data from Kusto ---
try:
    print(f"Reading from Kusto query...")

    kusto_query = f"""
    let lastRunParam = toscalar(DeleteLastRunTime | summarize max(LastRunTime));
    let startTimeFilter = iff(isnull(lastRunParam), ago(100d), lastRunParam);
    AlertLogs
    | where ingestion_time() >= startTimeFilter
    | where ingestion_time() < datetime('{execution_start_time}')
    | summarize arg_max(ingestion_time(), WorkspaceName, wstime, data_itemName, AlertStatus, data_itemKind) by WorkspaceId, data_itemId
    | where AlertStatus in ("EmailSent", "NoEmail")
    | project WorkspaceName, WorkspaceId, wstime, data_itemKind, data_itemName, data_itemId
    """
    
    # Read using Spark Kusto connector
    df_alerts = spark.read \
        .format("com.microsoft.kusto.spark.synapse.datasource") \
        .option("kustoCluster", kusto_cluster_uri) \
        .option("kustoDatabase", kusto_database) \
        .option("kustoQuery", kusto_query) \
        .option("accessToken", kusto_token) \
        .load()
    
    # Collect to driver for iteration (Assuming volume is manageable for alerts)
    # The SQL for AlertLogs has columns: WorkspaceId, data_itemId
    alerts_list = df_alerts.select("WorkspaceId", "WorkspaceName", "wstime", "data_itemKind", "data_itemName", "data_itemId").collect()
    
    print(f"Found {len(alerts_list)} alerts to process.")
    
except Exception as e:
    print(f"Error reading from Kusto: {e}")
    alerts_list = []

StatementMeta(, f675129f-c741-4a05-934f-ca0367a2c63c, 18, Finished, Available, Finished, False)

Reading from Kusto query...
Found 0 alerts to process.


In [17]:
# --- Main Loop ---
from collections import defaultdict

results_to_log = []
exceptions_to_log = []
deferred_deletion_list = []

if not alerts_list:
    print("No alerts to process.")
elif not fabric_token:
    print("Aborting Deletion Process: No Fabric token available.")
elif not kusto_token:
    print("Aborting Deletion Process: No Kusto token available.")
else:
    print(f"Starting deletion loop for {len(alerts_list)} items...")
    
    for row in alerts_list:
        item_id = row['data_itemId']
        ws_id = row['WorkspaceId']
        ws_name = row['WorkspaceName']
        item_kind = row['data_itemKind']
        
        print(f"Deleting '{row['data_itemName']}' ({item_id}) [{item_kind}] in ws: {ws_name} ({ws_id})")
        status, code, desc = delete_fabric_item(ws_id, item_id, item_kind)
        
        # Check for retry condition: 400 + UnknownError (potential dependency issue)
        # We defer these to the end, hoping dependent items are deleted in the first pass
        if code == "400" and "UnknownError" in desc:
             print(f"DEFERRING: Item {item_id} returned 400 UnknownError (possible dependency). Saving for retry.")
             deferred_deletion_list.append(row)
             continue

        # Logic: 200/204 (DeletedItem) and 404 (File Not Found) go to standard AlertLogs
        if status == "DeletedItem" or "404" in status:
            results_to_log.append({
                "WorkspaceName": ws_name,
                "WorkspaceId": ws_id,
                "wstime": row['wstime'],
                "data_itemKind": item_kind,
                "data_itemName": row['data_itemName'],
                "data_itemId": item_id,
                "AlertStatus": status
            })
        else:
            # Errors (401, 500, Exceptions) go to DeleteException
            exceptions_to_log.append({
                "WorkspaceName": ws_name,
                "WorkspaceId": ws_id,
                "wstime": row['wstime'],
                "data_itemKind": item_kind,
                "data_itemName": row['data_itemName'],
                "data_itemId": item_id,
                "AlertStatus": status,
                "ErrorCode": code,
                "Description": desc
            })

    # --- Retry Deferred Items ---
    if deferred_deletion_list:
        print(f"\n--- Retrying {len(deferred_deletion_list)} deferred items after initial pass ---")
        for row in deferred_deletion_list:
            item_id = row['data_itemId']
            ws_id = row['WorkspaceId']
            ws_name = row['WorkspaceName']
            item_kind = row['data_itemKind']
            
            print(f"RETRY Deleting '{row['data_itemName']}' ({item_id}) [{item_kind}]")
            status, code, desc = delete_fabric_item(ws_id, item_id, item_kind)
            
            # Treat result normally now (no second deferral)
            if status == "DeletedItem" or "404" in status:
                results_to_log.append({
                    "WorkspaceName": ws_name,
                    "WorkspaceId": ws_id,
                    "wstime": row['wstime'],
                    "data_itemKind": item_kind,
                    "data_itemName": row['data_itemName'],
                    "data_itemId": item_id,
                    "AlertStatus": status
                })
            else:
                exceptions_to_log.append({
                    "WorkspaceName": ws_name,
                    "WorkspaceId": ws_id,
                    "wstime": row['wstime'],
                    "data_itemKind": item_kind,
                    "data_itemName": row['data_itemName'],
                    "data_itemId": item_id,
                    "AlertStatus": status,
                    "ErrorCode": code,
                    "Description": desc
                })

    print(f"Deletion processing complete. Success/404: {len(results_to_log)}, Exceptions: {len(exceptions_to_log)}")

StatementMeta(, f675129f-c741-4a05-934f-ca0367a2c63c, 19, Finished, Available, Finished, False)

No alerts to process.


In [18]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, TimestampType

# --- Log Standard Results to Kusto (AlertLogs) ---
if 'results_to_log' in locals() and results_to_log:
    print(f"Logging {len(results_to_log)} entries to AlertLogs...")
    
    log_rows = [Row(**r) for r in results_to_log]
    log_df = spark.createDataFrame(log_rows)
    
    try:
        log_df.write \
            .format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", kusto_cluster_uri) \
            .option("kustoDatabase", kusto_database) \
            .option("kustoTable", "AlertLogs") \
            .option("accessToken", kusto_token) \
            .option("tableCreateOptions", "CreateIfNotExist") \
            .mode("Append") \
            .save()
        print("Successfully logged alerts status to AlertLogs.")
    except Exception as e:
        print(f"Failed to log to AlertLogs: {e}")
else:
    print("No standard results to log.")


# --- Log Exceptions to Kusto (DeleteException) ---
if 'exceptions_to_log' in locals() and exceptions_to_log:
    print(f"Logging {len(exceptions_to_log)} exceptions to DeleteException...")
    
    exc_rows = [Row(**r) for r in exceptions_to_log]
    exc_df = spark.createDataFrame(exc_rows)
    
    try:
        exc_df.write \
            .format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", kusto_cluster_uri) \
            .option("kustoDatabase", kusto_database) \
            .option("kustoTable", "DeleteException") \
            .option("accessToken", kusto_token) \
            .option("tableCreateOptions", "CreateIfNotExist") \
            .mode("Append") \
            .save()
        print("Successfully logged exceptions to DeleteException.")
    except Exception as e:
        print(f"Failed to log to DeleteException: {e}")
else:
    print("No exceptions to log.")


# --- Update Last Run Time ---
# We update the watermark table 'DeleteLastRunTime' with the execution start time.
if 'execution_start_time' in locals():
    print(f"Updating DeleteLastRunTime watermark to {execution_start_time}...")
    try:
        # Create DataFrame with single row
        time_data = [(execution_start_time,)]
        time_schema = StructType([StructField("LastRunTime", TimestampType(), False)])
        
        time_df = spark.createDataFrame(time_data, schema=time_schema)
        
        time_df.write \
            .format("com.microsoft.kusto.spark.synapse.datasource") \
            .option("kustoCluster", kusto_cluster_uri) \
            .option("kustoDatabase", kusto_database) \
            .option("kustoTable", "DeleteLastRunTime") \
            .option("accessToken", kusto_token) \
            .option("tableCreateOptions", "CreateIfNotExist") \
            .mode("Append") \
            .save()
        print("Successfully updated watermark.")
    except Exception as e:
        print(f"Failed to update LastRunTime: {e}")

StatementMeta(, f675129f-c741-4a05-934f-ca0367a2c63c, 20, Finished, Available, Finished, False)

No standard results to log.
No exceptions to log.
Updating DeleteLastRunTime watermark to 2026-03-04 14:20:28.767262+00:00...
Successfully updated watermark.
